[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anonymous-submission/multimodal-ML/blob/main/notebooks/interactive_linear_experiment_colab.ipynb)

# Interactive Demo: CA vs CP Under the Spiked Covariance Model

Explore when **Cross-Alignment (CA, contrastive)** vs **Cross-Prediction (CP, predictive)** recovers shared signal in multimodal data, predicted by the Spiked Covariance Model.

> **▶ Run the Setup cell first**, then interact with the two sections below.

In [ ]:
# @title ⚙ Setup — run this cell first {display-mode: "form"}
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.colors import ListedColormap
import ipywidgets as widgets
from IPython.display import display, clear_output
from dataclasses import dataclass

# ── Generative model ──────────────────────────────────────────────────────────
@dataclass
class SyntheticData:
    X: np.ndarray; Y: np.ndarray; Y_signal: np.ndarray
    Q_x: np.ndarray; Q_y: np.ndarray
    signal_subspace_x: np.ndarray; signal_subspace_y: np.ndarray
    kappas: np.ndarray; S_xx: np.ndarray; S_yy: np.ndarray; S_xy: np.ndarray

def _random_orthogonal(d, rng):
    H = rng.standard_normal((d, d)); Q, R = np.linalg.qr(H)
    return Q @ np.diag(np.sign(np.diag(R)))

def generate_multimodal_data(n, d, k, kappas, gxs, gys, gxn, gyn, etas, seed=None):
    rng = np.random.default_rng(seed)
    kappas, gxs, gys, gxn, gyn, etas = [np.asarray(a, float) for a in (kappas, gxs, gys, gxn, gyn, etas)]
    Q_x, Q_y = _random_orthogonal(d, rng), _random_orthogonal(d, rng)
    signal = rng.standard_normal((n, k)) * kappas
    xc = np.concatenate([signal + rng.standard_normal((n, k)) * np.sqrt(gxs),
                         rng.standard_normal((n, d - k)) * np.sqrt(gxn)], axis=1)
    noise_yn = np.zeros((n, d - k))
    noise_xn = np.zeros((n, d - k))
    for j in range(d - k):
        cov = np.array([[gxn[j], etas[j]], [etas[j], gyn[j]]])
        s = rng.multivariate_normal([0, 0], cov, n)
        noise_xn[:, j] = s[:, 0]; noise_yn[:, j] = s[:, 1]
    xc = np.concatenate([signal + rng.standard_normal((n, k)) * np.sqrt(gxs), noise_xn], axis=1)
    yc = np.concatenate([signal + rng.standard_normal((n, k)) * np.sqrt(gys), noise_yn], axis=1)
    X, Y = xc @ Q_x.T, yc @ Q_y.T
    S_xx = Q_x @ np.diag(np.concatenate([kappas**2 + gxs, gxn])) @ Q_x.T
    S_yy = Q_y @ np.diag(np.concatenate([kappas**2 + gys, gyn])) @ Q_y.T
    S_xy = Q_x @ np.diag(np.concatenate([kappas**2, etas])) @ Q_y.T
    return SyntheticData(X=X, Y=Y, Y_signal=signal @ Q_y[:, :k].T,
                         Q_x=Q_x, Q_y=Q_y,
                         signal_subspace_x=Q_x[:, :k], signal_subspace_y=Q_y[:, :k],
                         kappas=kappas, S_xx=S_xx, S_yy=S_yy, S_xy=S_xy)

def compute_sample_covariances(X, Y):
    n = X.shape[0]; return X.T @ X / n, Y.T @ Y / n, X.T @ Y / n

# ── Theory ────────────────────────────────────────────────────────────────────
@dataclass
class RecoveryPredictions:
    rho: np.ndarray; nu: np.ndarray; delta_ca: float
    tau: np.ndarray; xi: np.ndarray; delta_cp: float
    ca_recovers: bool; cp_recovers: bool

def compute_recovery_predictions(kappas, gxs, gys, gxn, gyn, etas):
    kappas, gxs, gys, gxn, gyn, etas = [np.asarray(a, float) for a in (kappas, gxs, gys, gxn, gyn, etas)]
    rho = kappas**2 / np.sqrt((kappas**2 + gxs) * (kappas**2 + gys))
    nu  = etas / np.sqrt(gxn * gyn + 1e-30)
    tau = kappas**2 / np.sqrt(kappas**2 + gxs)
    xi  = etas / np.sqrt(gxn + 1e-30)
    max_nu, max_xi = np.max(nu), np.max(xi)
    delta_ca = float(np.min(rho) / max_nu)  if max_nu > 1e-15 else np.inf
    delta_cp = float(np.min(tau) / max_xi)  if max_xi > 1e-15 else np.inf
    return RecoveryPredictions(rho=rho, nu=nu, delta_ca=delta_ca,
                               tau=tau, xi=xi, delta_cp=delta_cp,
                               ca_recovers=delta_ca > 1, cp_recovers=delta_cp > 1)

# ── Solvers ───────────────────────────────────────────────────────────────────
def _mat_sqrt_inv(M):
    v, Q = np.linalg.eigh(M)
    return Q @ np.diag(1 / np.sqrt(np.maximum(v, 1e-12))) @ Q.T

@dataclass
class CASolution:
    W: np.ndarray; V: np.ndarray; canonical_correlations: np.ndarray

@dataclass
class CPSolution:
    B: np.ndarray; E: np.ndarray; singular_values: np.ndarray

def solve_ca(S_xx, S_yy, S_xy, k):
    Sx, Sy = _mat_sqrt_inv(S_xx), _mat_sqrt_inv(S_yy)
    P, phi, Qt = np.linalg.svd(Sx @ S_xy @ Sy, full_matrices=False)
    Pk, Qk, phik = P[:, :k], Qt[:k].T, phi[:k]
    fi = np.diag(1 / np.sqrt(np.maximum(phik, 1e-12)))
    return CASolution(W=fi @ Pk.T @ Sx, V=fi @ Qk.T @ Sy, canonical_correlations=phik)

def solve_cp(S_xx, S_yy, S_xy, k):
    Sx = _mat_sqrt_inv(S_xx)
    U, sig, Vt = np.linalg.svd(S_xy.T @ Sx, full_matrices=False)
    Uk, sk, Vk = U[:, :k], sig[:k], Vt[:k]
    return CPSolution(B=Uk @ np.diag(sk) @ Vk @ Sx, E=Vk @ Sx, singular_values=sk)

def recovered_subspace(W, S_xx):
    return np.linalg.svd(W, full_matrices=False)[2].T

def subspace_distance(U_true, U_rec):
    Q1, _ = np.linalg.qr(U_true); Q2, _ = np.linalg.qr(U_rec)
    k = min(U_true.shape[1], U_rec.shape[1])
    cos = np.linalg.svd(Q1[:, :k].T @ Q2[:, :k], compute_uv=False)
    return 1.0 - np.sum(np.minimum(cos, 1.0)**2) / k

# ── Plot style ────────────────────────────────────────────────────────────────
plt.rcParams.update({'font.family': 'serif', 'mathtext.fontset': 'cm',
                     'axes.labelsize': 15, 'axes.titlesize': 16,
                     'axes.labelweight': 'bold', 'axes.titleweight': 'bold',
                     'xtick.labelsize': 12, 'ytick.labelsize': 12,
                     'axes.linewidth': 1.3, 'xtick.direction': 'in', 'ytick.direction': 'in'})

C_CA = '#4A90C4'; C_CP = '#E07B54'
C_NEITHER = '#F2F0EB'; C_CA_ONLY = '#4A90C4'; C_CP_ONLY = '#E07B54'; C_BOTH = '#7B6D8E'
C_CA_LINE = '#1A3A5C'; C_CP_LINE = '#8B3A1E'

print("✓ Setup complete.")


## 1 · Phase Diagram

The colored regions show which method recovers the signal subspace.
Increase **γ̃_y** (target nuisance variance) to observe the **Variance Trap**: the CA-only region expands as nuisance dominates CP.

In [ ]:
# @title Phase Diagram
_style  = {'description_width': '170px'}
_layout = widgets.Layout(width='440px')

def _sl(desc, val, lo, hi, step):
    return widgets.FloatSlider(value=val, min=lo, max=hi, step=step,
                               description=desc, style=_style, layout=_layout)

w_gx   = _sl('\u03b3\u2093  source noise',         1.00,  0.01, 10.0, 0.10)
w_gy   = _sl('\u03b3_y  target signal noise',  0.05,  0.01,  5.0, 0.05)
w_gty  = _sl('\u03b3\u0303_y target nuisance', 5.00,  0.01, 20.0, 0.25)
w_kmax = _sl('\u03ba_max',                          3.00,  1.00,  8.0, 0.50)

out_pd = widgets.Output()

def _update_pd(*_):
    gx, gy, gty, kmax = w_gx.value, w_gy.value, w_gty.value, w_kmax.value
    res = 600
    kk = np.linspace(0, kmax, res)
    nu = np.linspace(0, 1.0, res)
    K, N = np.meshgrid(kk, nu)

    ca_ok = N < K**2 / np.sqrt((K**2 + gx) * (K**2 + gy))
    cp_ok = N < K**2 / (np.sqrt(K**2 + gx) * np.sqrt(gty))
    region = np.where(ca_ok & cp_ok, 3, np.where(ca_ok, 1, np.where(cp_ok, 2, 0)))

    with out_pd:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(7, 5.5))
        cmap = ListedColormap([C_NEITHER, C_CA_ONLY, C_CP_ONLY, C_BOTH])
        ax.imshow(region, origin='lower', aspect='auto',
                  extent=[0, kmax, 0, 1.0], cmap=cmap, vmin=0, vmax=3,
                  interpolation='bilinear')

        kline = np.linspace(0.01, kmax, 800)
        nu_ca = kline**2 / np.sqrt((kline**2 + gx) * (kline**2 + gy))
        nu_cp = np.clip(kline**2 / (np.sqrt(kline**2 + gx) * np.sqrt(gty)), 0, 1.0)
        ax.plot(kline, nu_ca, color=C_CA_LINE, lw=2.8, label=r'$\Delta_{\rm CA}=1$')
        ax.plot(kline, nu_cp, color=C_CP_LINE, lw=2.8, ls='--', dashes=(6, 3),
                label=r'$\Delta_{\rm CP}=1$')

        legend_els = [
            Patch(facecolor=C_NEITHER, ec='#999', lw=1, label='Neither'),
            Patch(facecolor=C_CA_ONLY, ec='#999', lw=1, label='CA only'),
            Patch(facecolor=C_CP_ONLY, ec='#999', lw=1, label='CP only'),
            Patch(facecolor=C_BOTH,    ec='#999', lw=1, label='Both'),
            plt.Line2D([0],[0], color=C_CA_LINE, lw=2.5, label=r'$\Delta_{\rm CA}=1$'),
            plt.Line2D([0],[0], color=C_CP_LINE, lw=2.5, ls='--',
                       label=r'$\Delta_{\rm CP}=1$'),
        ]
        ax.legend(handles=legend_els, loc='upper right', fontsize=11, framealpha=0.9)
        ax.set_xlabel(r'Signal strength $\kappa$')
        ax.set_ylabel(r'Nuisance correlation $\nu$')
        ax.set_xlim(0, kmax); ax.set_ylim(0, 1.0)
        ax.set_title(
            rf'$\gamma_x={gx:.2f},\;\gamma_y={gy:.2f},\;\tilde{{\gamma}}_y={gty:.2f}$',
            pad=10)
        plt.tight_layout(); display(fig); plt.close(fig)

for w in [w_gx, w_gy, w_gty, w_kmax]:
    w.observe(_update_pd, names='value')

display(widgets.VBox([w_gx, w_gy, w_gty, w_kmax]), out_pd)
_update_pd()


## 2 · Noise Correlation Sweep

Sweep **ν** (cross-modal nuisance correlation). Higher ν hurts CA but not CP — tracing the CA phase boundary from left to right.

In [ ]:
# @title Noise Correlation Sweep
_style2  = {'description_width': '170px'}
_layout2 = widgets.Layout(width='440px')

def _sl2(desc, val, lo, hi, step, is_int=False):
    cls = widgets.IntSlider if is_int else widgets.FloatSlider
    return cls(value=val, min=lo, max=hi, step=step,
               description=desc, style=_style2, layout=_layout2)

w_kap  = _sl2('\u03ba  signal strength',           1.50, 0.10, 5.0,  0.10)
w_gx2  = _sl2('\u03b3\u2093  source noise',        1.00, 0.01, 10.0, 0.10)
w_gy2  = _sl2('\u03b3_y  target signal noise', 0.05, 0.01, 5.0,  0.05)
w_gty2 = _sl2('\u03b3\u0303_y target nuisance',5.00, 0.01, 20.0, 0.50)
w_n2   = _sl2('n  samples',                        5000, 1000, 20000, 1000, is_int=True)

out_nu = widgets.Output()

def _update_nu(*_):
    kappa, gx, gy, gty, n = (w_kap.value, w_gx2.value, w_gy2.value,
                               w_gty2.value, w_n2.value)
    d, k = 20, 3
    nu_range = np.linspace(0, 0.98, 30)
    deltas_ca, deltas_cp, dists_ca, dists_cp = [], [], [], []

    for nu_val in nu_range:
        kappas = np.full(k, kappa)
        gxs = np.full(k, gx);  gys = np.full(k, gy)
        gxn = np.full(d-k, 1.0); gyn = np.full(d-k, gty)
        etas = nu_val * np.sqrt(gxn * gyn)
        t = compute_recovery_predictions(kappas, gxs, gys, gxn, gyn, etas)
        deltas_ca.append(t.delta_ca); deltas_cp.append(t.delta_cp)

        data = generate_multimodal_data(n, d, k, kappas, gxs, gys, gxn, gyn, etas, seed=42)
        Sxx, Syy, Sxy = compute_sample_covariances(data.X, data.Y)
        ca = solve_ca(Sxx, Syy, Sxy, k); cp = solve_cp(Sxx, Syy, Sxy, k)
        dists_ca.append(subspace_distance(data.signal_subspace_x,
                                          recovered_subspace(ca.W, Sxx)))
        dists_cp.append(subspace_distance(data.signal_subspace_x,
                                          recovered_subspace(cp.E, Sxx)))

    with out_nu:
        clear_output(wait=True)
        fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
        fig.subplots_adjust(wspace=0.35, bottom=0.18, top=0.88)

        ax = axes[0]
        ax.plot(nu_range, deltas_ca, '-o', color=C_CA, lw=2.2, ms=4,
                label=r'$\Delta_{\rm CA}$')
        ax.plot(nu_range, deltas_cp, '--s', color=C_CP, lw=2.2, ms=4,
                label=r'$\Delta_{\rm CP}$')
        ax.axhline(1.0, color='k', ls=':', lw=1, alpha=0.5)
        ax.set_xlabel(r'Nuisance correlation $\nu$')
        ax.set_ylabel('Separation ratio')
        ax.set_title('Recovery Thresholds')
        ax.legend(fontsize=12)

        ax = axes[1]
        ax.plot(nu_range, dists_ca, '-o', color=C_CA, lw=2.2, ms=4, label='CA')
        ax.plot(nu_range, dists_cp, '--s', color=C_CP, lw=2.2, ms=4, label='CP')
        ax.set_xlabel(r'Nuisance correlation $\nu$')
        ax.set_ylabel('Subspace distance')
        ax.set_title('Signal Recovery  (0 = perfect)')
        ax.set_ylim(-0.05, 1.05)
        ax.legend(fontsize=12)

        display(fig); plt.close(fig)

for w in [w_kap, w_gx2, w_gy2, w_gty2, w_n2]:
    w.observe(_update_nu, names='value')

display(widgets.VBox([w_kap, w_gx2, w_gy2, w_gty2, w_n2]), out_nu)
_update_nu()
